In [7]:

import pandas as pd
import requests
from bs4 import BeautifulSoup

def collect_data():
    
    data = {
        'review_id': [1, 2, 3, 4, 5],
        'user_name': ['Youssef', 'Ahmed', 'Marwan', 'None', 'Yasser'],
        'text_content': [
            "The movie was absolutely amazing! 10/10",
            "Terrible experience, would not recommend.",
            "It was okay, but a bit too long.",
            "!!! ERROR !!! No content here.",
            "Loved the acting but hated the plot."
        ],
        'rating': [10, 2, 5, 0, 7],
        'source': ['IMDb', 'IMDb', 'Letterboxd', 'IMDb', 'RottenTomatoes']
    }
    df = pd.DataFrame(data)
    df.to_csv('movie_reviews_dataset.csv', index=False)
    return df

df = collect_data()
print("Dataset Created!")

Dataset Created!


In [ ]:

def clean_data(df):
    
    df['user_name'] = df['user_name'].replace('none', 'Anonymous')
    
   
    df['text_content'] = df['text_content'].str.strip().str.lower()
    
   
    df = df[~df['text_content'].str.contains("error")]
    
    
    df['sentiment'] = df['rating'].apply(lambda x: 'Positive' if x >= 7 else ('Neutral' if x >= 5 else 'Negative'))
    
    return df
# to delete warning
df_cleaned = df[~df['text_content'].str.contains("error")].copy()

In [9]:
class TrustGuardValidator:
    def __init__(self, schema_config):
        self.schema = schema_config

    def validate(self, data_row):
       
        errors = []
        text = data_row.get('text_content', '')
        
        
        if len(text) < self.schema.get('min_length', 1):
            errors.append("Text too short")
            
        
        if self.schema.get('check_keywords'):
            keywords = ['movie', 'acting', 'plot', 'recommend', 'experience']
            if not any(word in text for word in keywords):
                errors.append("Missing context keywords")
        
       
        rating = data_row.get('rating', 0)
        if not (self.schema.get('min_rating') <= rating <= self.schema.get('max_rating')):
            errors.append(f"Rating {rating} out of range")

        return {"is_valid": len(errors) == 0, "errors": errors}


my_schema = {
    'min_length': 10,
    'check_keywords': True,
    'min_rating': 1,
    'max_rating': 10
}

validator = TrustGuardValidator(my_schema)


df_cleaned['validation_results'] = df_cleaned.apply(lambda row: validator.validate(row), axis=1)